# NB03: External validation

Applies fitted models to holdout datasets:
1. AusMicrobiome + NGSA (Australian soils)
2. EMP (Earth Microbiome Project)
3. SPIRE (soil microbiome compendium)

**Outputs**
- `data/holdout_results.csv` — transfer RMSE per holdout × model × target
- `data/holdout_feature_matrices/` — feature matrices for each holdout set

**NOTE**: Holdout data access must be confirmed before running. Update paths below.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
HOLDOUT_DIR = DATA_DIR / 'holdout_feature_matrices'
HOLDOUT_DIR.mkdir(exist_ok=True)

targets = ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']

# Spark session
try:
    spark
except NameError:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
print('Spark ready:', spark.version)

# Load training feature matrix for refitting final models
train_fm = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
print(f'Training feature matrix: {train_fm.shape}')

## 1. Fit final models on full training set

For holdout evaluation, fit final models on ALL training data (no CV).

In [ ]:
from modelling import build_xgboost, build_ridge, get_features, _drop_nan_rows

final_models = {}

for target in [t for t in targets if t in train_fm.columns]:
    y = train_fm[target]
    for model_name, model_type in [('M2', 'xgboost'), ('M3', 'xgboost'), ('M4', 'xgboost'), ('B0', None)]:
        X = get_features(train_fm, model_name)
        Xc, yc = _drop_nan_rows(X, y)
        if model_name == 'B0':
            final_models[(target, model_name)] = float(yc.mean())
        elif model_type == 'xgboost':
            m = build_xgboost()
            m.fit(Xc, yc)
            final_models[(target, model_name)] = m
    print(f'  Fit final models for {target}')

print('Final models fit (M2, M3, M4, B0).')

## 2. Holdout helper function

In [ ]:
from cwm_utils import load_otu_bridge, load_genus_densities, compute_cwm
from soilgrids_api import SoilGridsClient
from env_utils import GEOROC_METAL_COLS, merge_ph
from modelling import rmse, get_features, ENV_FEATURES

bridge = load_otu_bridge(DATA_DIR / 'otu_pangenome_link_v2.csv')
densities = load_genus_densities(genus_trait_path=DATA_DIR / 'genus_trait_table.csv')
sg_client = SoilGridsClient(cache_path=str(DATA_DIR / 'soilgrids_cache.json'))


def evaluate_holdout(
    genus_ra_wide: pd.DataFrame,   # sample × genus relative abundance
    metal_targets: pd.DataFrame,   # sample × metal target log columns
    coords: pd.DataFrame,          # sample × [lat, lon, ph_insitu?]
    holdout_name: str,
) -> pd.DataFrame:
    """Compute holdout feature matrix and evaluate final models."""

    # CWM
    cwm = compute_cwm(genus_ra_wide, densities)
    print(f'{holdout_name}: CWM coverage: {cwm["coverage_fraction"].mean():.2f}')

    # SoilGrids via API
    api_df = sg_client.batch_query(coords, lat_col='lat', lon_col='lon', id_col=None)
    api_df.index = coords.index

    # Env features
    cheap = coords.copy()
    cheap = cheap.join(api_df, how='left')
    cheap['ph'] = cheap.apply(merge_ph, axis=1)
    cheap['ph_source'] = 'soilgrids'
    if 'ph_insitu' in cheap.columns:
        cheap.loc[cheap['ph_insitu'].notna(), 'ph_source'] = 'insitu'

    fm = cheap.join(cwm, how='inner').join(metal_targets, how='inner')
    fm.to_parquet(HOLDOUT_DIR / f'{holdout_name}_feature_matrix.parquet')

    # Evaluate
    records = []
    for target in [t for t in targets if t in fm.columns]:
        y = fm[target]
        for model_name in ['B0', 'M4', 'M2']:
            model = final_models.get((target, model_name))
            if model is None:
                continue
            if model_name == 'B0':
                preds = np.full(len(y), model)
            else:
                X = get_features(fm, model_name)
                valid = ~X.isna().any(axis=1) & y.notna()
                preds = np.full(len(y), np.nan)
                if valid.sum() > 0:
                    preds[valid] = model.predict(X[valid])
            valid = ~np.isnan(preds) & y.notna()
            if valid.sum() > 0:
                records.append({
                    'holdout': holdout_name,
                    'model': model_name,
                    'target': target,
                    'n': valid.sum(),
                    'rmse': rmse(y[valid].values, preds[valid]),
                    'cwm_coverage_mean': cwm['coverage_fraction'].loc[fm.index].mean(),
                })
    return pd.DataFrame(records)

print('Holdout helper defined.')

## 3. AusMicrobiome + NGSA

Data sources:
- `BASE_16S_OTU.csv.gz` — OTU counts (91,929 OTUs × 1,023 samples; OTU_Id × numeric sample IDs)
- `BASE_16S_taxonomy.csv` — OTU_Id → genus string (SILVA format: `g__Bradyrhizobium`)
- `aus_sample_ngsa.csv` — 1,663 samples with NGSA metal concentrations + field pH; Sample_ID suffix matches OTU numeric column IDs (1,019 samples overlap)

Feature assembly:
- `ph` — from `ngsa_field_pH` (fallback: SoilGrids API `ph_h2o`)
- `clay_pct` — SoilGrids REST API
- `mob_*` — CSU metal mobility grid via Spark (`get_csu_mobility_features`)
- `water_content, ndvi, elevation_m, temp_K, precip_mm` — NOT available for non-MicrobeAtlas samples; left as NaN; XGBoost uses learned default-direction split routing for missing values

Models evaluated: B0, M3, M4, M2

In [ ]:
import gzip
from env_utils import get_csu_mobility_features
from modelling import rmse, get_features, ENV_FEATURES

MICRO_DIR = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/aus_microbiome')

# ── 1. Taxonomy: OTU → genus ─────────────────────────────────────────────────
print('Loading AusMicrobiome taxonomy...')
tax = pd.read_csv(MICRO_DIR / 'BASE_16S_taxonomy.csv')
tax.columns = ['OTU_Id', 'genus_str']
tax['genus_lower'] = (
    tax['genus_str']
    .str.replace('g__', '', regex=False)
    .str.strip()
    .str.lower()
)
tax.loc[
    tax['genus_lower'].isin(['', 'unclassified']) | tax['genus_lower'].isna(),
    'genus_lower',
] = None
otu_to_genus = tax.dropna(subset=['genus_lower']).set_index('OTU_Id')['genus_lower'].to_dict()
print(f'  {len(otu_to_genus):,} OTUs with genus assignment')

# ── 2. OTU counts → genus RA ─────────────────────────────────────────────────
print('Loading OTU counts...')
with gzip.open(MICRO_DIR / 'BASE_16S_OTU.csv.gz', 'rt') as f:
    otu_counts = pd.read_csv(f, index_col=0)
print(f'  {otu_counts.shape[0]:,} OTUs × {otu_counts.shape[1]:,} samples')

otu_counts.index = otu_counts.index.map(otu_to_genus)
otu_counts = otu_counts[otu_counts.index.notna()]
genus_counts = otu_counts.groupby(level=0).sum()
genus_counts.index = genus_counts.index.str.lower().str.strip()
genus_ra = genus_counts.T
totals = genus_ra.sum(axis=1).replace(0.0, np.nan)
genus_ra = genus_ra.div(totals, axis=0)
print(f'  Genus RA: {genus_ra.shape[0]:,} samples × {genus_ra.shape[1]:,} genera')

# ── 3. Join with NGSA metals ──────────────────────────────────────────────────
print('Joining with NGSA...')
ngsa = pd.read_csv(MICRO_DIR / 'aus_sample_ngsa.csv')
ngsa['otu_key'] = ngsa['Sample_ID'].str.split('/').str[-1]
ngsa = ngsa.set_index('otu_key')

common = genus_ra.index.intersection(ngsa.index)
print(f'  Common samples: {len(common)}')
genus_ra_c = genus_ra.loc[common]
ngsa_c = ngsa.loc[common]

# ── 4. CWM ────────────────────────────────────────────────────────────────────
print('Computing CWM...')
from cwm_utils import compute_cwm
cwm_aus = compute_cwm(genus_ra_c, densities)
print(f'  Coverage: {cwm_aus["coverage_fraction"].mean():.3f} mean, '
      f'{cwm_aus["low_coverage_flag"].mean():.1%} flagged')

# ── 5. CSU mobility features via Spark ───────────────────────────────────────
print('Getting CSU mobility features via Spark...')
aus_coords = pd.DataFrame({
    'lat': pd.to_numeric(ngsa_c['latitude'], errors='coerce').values,
    'lon': pd.to_numeric(ngsa_c['longitude'], errors='coerce').values,
}, index=common)
aus_coords.index.name = 'sample_id'

mob_df = get_csu_mobility_features(spark, aus_coords)
n_mob = mob_df.dropna(how='all').shape[0]
print(f'  CSU mobility matched: {n_mob:,} / {len(common):,}')

# ── 6. Build holdout feature matrix ──────────────────────────────────────────
# Features by source:
#   ph          → ngsa_field_pH (available)
#   mob_*       → CSU Spark join (available)
#   clay_pct    → NaN (SoilGrids API too slow for 1k+ uncached samples)
#   water_content, ndvi, elevation_m, temp_K, precip_mm → NaN (GEE table is
#       MicrobeAtlas-specific; ERA5/NDVI not queried here)
# XGBoost routes NaN samples via the learned default split direction at each node.
aus_fm = pd.DataFrame(index=common, columns=ENV_FEATURES, dtype=float)

if 'ngsa_field_pH' in ngsa_c.columns:
    aus_fm['ph'] = pd.to_numeric(ngsa_c['ngsa_field_pH'], errors='coerce').values

for col in ['mob_cu', 'mob_pb', 'mob_as', 'mob_cd', 'mob_cr', 'mob_hg']:
    if col in mob_df.columns:
        aus_fm[col] = mob_df[col].values

cwm_cols = [c for c in cwm_aus.columns if c.startswith('CWM_')]
aus_fm = aus_fm.join(cwm_aus[cwm_cols])

metal_map = {
    'ngsa_Cu_ppm': 'log_Cu_ppm',
    'ngsa_Zn_ppm': 'log_Zn_ppm',
    'ngsa_Pb_ppm': 'log_Pb_ppm',
    'ngsa_Ni_ppm': 'log_Ni_ppm',
}
for src, dst in metal_map.items():
    if src in ngsa_c.columns:
        aus_fm[dst] = np.log1p(pd.to_numeric(ngsa_c[src], errors='coerce').values)

aus_fm['lat'] = aus_coords['lat'].values
aus_fm['lon'] = aus_coords['lon'].values
aus_fm.to_parquet(HOLDOUT_DIR / 'AusMicrobiome_NGSA_feature_matrix.parquet')

n_avail = {col: int(aus_fm[col].notna().sum()) for col in ENV_FEATURES if col in aus_fm.columns}
print('Feature availability:', n_avail)

# ── 7. Evaluate B0, M3, M4, M2 ───────────────────────────────────────────────
# H6 requires M2 and M4 to be evaluated on the SAME sample subset so the
# M2/M4 RMSE ratio is computed on identical samples.  Without this, M2
# uses ~730 samples (CWM never all-NaN) and M4 uses ~480 samples (env
# features sparse), making the ratio comparison invalid (C1 fix).
aus_records = []
for target in targets:
    if target not in aus_fm.columns:
        continue
    y = aus_fm[target]
    valid_y = y.notna()

    # Pre-compute M2/M4 intersection for the H6 ratio comparison
    X_m2_all = get_features(aus_fm, 'M2')
    X_m4_all = get_features(aus_fm, 'M4')
    valid_m2m4 = (
        valid_y
        & ~X_m2_all.isna().all(axis=1)
        & ~X_m4_all.isna().all(axis=1)
    )
    n_m2_only = int((valid_y & ~X_m2_all.isna().all(axis=1)).sum())
    n_m4_only = int((valid_y & ~X_m4_all.isna().all(axis=1)).sum())
    print(f'  {target}: M2∩M4 valid = {int(valid_m2m4.sum())} '
          f'(M2-only = {n_m2_only}, M4-only = {n_m4_only})')

    for model_name in ['B0', 'M3', 'M4', 'M2']:
        model = final_models.get((target, model_name))
        if model is None:
            continue

        if model_name == 'B0':
            n = int(valid_y.sum())
            if n < 5:
                continue
            aus_records.append({
                'holdout': 'AusMicrobiome_NGSA', 'model': model_name, 'target': target,
                'n': n,
                'rmse': rmse(y[valid_y].values, np.full(n, model)),
                'cwm_coverage_mean': float(cwm_aus['coverage_fraction'].mean()),
            })
        elif model_name in ('M2', 'M4'):
            # Use intersection so H6 M2/M4 ratio compares identical samples
            valid = valid_m2m4
            n = int(valid.sum())
            if n < 5:
                continue
            X = get_features(aus_fm, model_name)
            preds = model.predict(X[valid])
            aus_records.append({
                'holdout': 'AusMicrobiome_NGSA', 'model': model_name, 'target': target,
                'n': n,
                'rmse': rmse(y[valid].values, preds),
                'cwm_coverage_mean': float(cwm_aus['coverage_fraction'].mean()),
            })
        else:  # M3
            X = get_features(aus_fm, model_name)
            valid = valid_y & ~X.isna().all(axis=1)
            n = int(valid.sum())
            if n < 5:
                continue
            preds = model.predict(X[valid])
            aus_records.append({
                'holdout': 'AusMicrobiome_NGSA', 'model': model_name, 'target': target,
                'n': n,
                'rmse': rmse(y[valid].values, preds),
                'cwm_coverage_mean': float(cwm_aus['coverage_fraction'].mean()),
            })

aus_results = pd.DataFrame(aus_records)
if not aus_results.empty:
    print(aus_results.pivot_table(index='model', columns='target', values='rmse').round(4))
    # Print intersection sample counts for audit
    n_by_target = aus_results[aus_results['model'] == 'M2'].set_index('target')['n']
    print('M2/M4 intersection n per target:', n_by_target.to_dict())
else:
    print('No results — check data paths')
print('AusMicrobiome+NGSA complete.')


## 4. EMP

In [ ]:
# Placeholder — update with actual EMP data paths
# emp_genus_ra = ...
# emp_results = evaluate_holdout(emp_genus_ra, emp_metals, emp_coords, 'EMP')
print('EMP: data paths not yet configured — skip')
emp_results = pd.DataFrame()

## 5. SPIRE

In [ ]:
# Placeholder — update with actual SPIRE data paths
print('SPIRE: data paths not yet configured — skip')
spire_results = pd.DataFrame()

In [ ]:
# Combine and save
holdout_results = pd.concat([aus_results, emp_results, spire_results], ignore_index=True)
if not holdout_results.empty:
    holdout_results.to_csv(DATA_DIR / 'holdout_results.csv', index=False)
    print(holdout_results.pivot_table(
        index=['holdout', 'model'], columns='target', values='rmse'
    ).round(4))
else:
    print('No holdout results to save — update data paths and rerun.')